# Imports

## Libraries

In [ ]:
import sys; sys.path.append("..")

import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from sklearn.feature_selection import mutual_info_classif

## Data

In [ ]:
df = pd.read_parquet("../data/brfss2025_il_clean.parquet")
with open("../data/brfss2025_il_clean_columns.json") as f:
    meta = json.load(f)
    
cat_predictors, num_predictors, bin_predictors, predictor_groups = meta["cat_predictors"], meta["num_predictors"],meta["bin_predictors"], meta["predictor_groups"]
predictors = cat_predictors + num_predictors + bin_predictors
group_predictors = [c for cols in predictor_groups.values() for c in cols]

num_set, cat_set, bin_set = set(num_predictors), set(cat_predictors), set(bin_predictors)
# No column in more than one type list
assert len(predictors) == len(set(predictors)), {c for c in predictors if predictors.count(c) > 1}

# Type lists and groups dict contain the same names
assert set(predictors) == set(group_predictors), set(predictors) ^ set(group_predictors)

# All predictors in data
assert set(predictors) <= set(df.columns), set(predictors) - set(df.columns)

X = df[predictors]
y = df[meta["target"]]

## Functions

In [ ]:
def pick(group=None, kind=None):
    """
    Predictors filtered by group and/or type
    
    group: "demographic", "behavior", "health", or None for all groups
    kind: "num", "cat", "bin" or None for both
    """
    cols = predictor_groups[group] if group else predictors
    if kind == "num":
        return [c for c in cols if c in num_set]
    if kind == "cat":
        return [c for c in cols if c in cat_set]
    if kind == "bin":
        return [c for c in cols if c in bin_set]
    return list(cols)

# Missingness

In [ ]:
# Share of complete rows
print(f"Full row percentage:\t{(df[predictors].dropna().shape[0] / len(df) * 100):.2f}")

# Missingness in each column
missing = df.isna()
missing_pct = missing.mean().mul(100).sort_values(ascending=False)
missing_pct.name = "Percent missing"
print("\nMissingness by predictor:")
display(missing_pct.round(2))

# Target distribution in complete/not rows
complete = df[predictors].notna().all(axis=1)
print("Target dist for rows that are/aren't complete:")
display(df.groupby(complete)["DIABETES"].mean().round(4) * 100)


# Column Distributions

## Target distribution
Large class imbalance

In [ ]:
print(f"Positive class share:\t{df["DIABETES"].mean().round(4)}")

## Predictor summaries

### Categorical

In [ ]:
for col in cat_predictors + bin_predictors:
    display(df[col].value_counts(dropna=False, sort=False))

### Numerical

In [ ]:
summary = df[num_predictors].describe().T
summary["skew"] = df[num_predictors].skew()
display(summary.round(3))

Make histogram for rows with high skew
- all skew right
- log transform helps with fruit_daily and veg_daily

In [ ]:
high_skew_preds = summary.index[summary["skew"].abs() >= 1.5].tolist()
for col in high_skew_preds:
    df[col].plot(kind='hist', bins=10, edgecolor='black')
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.title(f"Distribution of {col}")
    plt.show()

    np.log1p(df[col]).plot(kind='hist', bins=10, edgecolor='black')
    plt.xlabel(f"Log(1 + {col})")
    plt.ylabel("Count")
    plt.title(f"Distribution of log transform {col}")
    plt.show()

## Continuous correlation
- Minor correlation within predictor groups

In [ ]:
corr = df[num_predictors].corr().round(2)
display(corr)

plt.figure(figsize=(9,7))
im = plt.imshow(corr, cmap="seismic", vmin=-1, aspect="auto")
plt.colorbar(im)
plt.xticks(np.arange(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(np.arange(len(corr.index)), corr.index)
plt.show()

# Predictor-Target relationships

## Categorical and Binary

In [ ]:
base_rate = df["DIABETES"].mean()

for col in cat_predictors + bin_predictors:
    stats = df.groupby(col, observed=True)["DIABETES"].agg(["mean", "size"])
    se = np.sqrt(stats["mean"] * (1 - stats["mean"]) / stats["size"])
    labels = [f"{lvl}\n(n={n})" for lvl, n in zip(stats.index, stats["size"])]

    fig, ax = plt.subplots()
    ax.bar(labels, stats["mean"], yerr=1.96 * se, capsize=4)
    ax.axhline(base_rate, ls="--", c="gray")
    ax.set(ylabel="Diabetes rate", title=col)
    plt.xticks(rotation=45, ha="right")
    plt.show()

## Numerical

In [ ]:
for col in num_predictors:
    rate = df.groupby(pd.qcut(df[col], q=10, duplicates="drop"), observed=True)["DIABETES"].mean()
    ax = rate.plot(marker="o")
    ax.axhline(base_rate, ls="--", c="gray")
    ax.set(ylabel="Diabetes rate", title=f"{col} (deciles)")
    plt.xticks(rotation=45)
    plt.show()

## Mutual information

In [ ]:

mi = {}
for col in predictors:
    sub = df[[col, "DIABETES"]].dropna()
    x = sub[col].cat.codes if col in cat_set else sub[col]
    mi[col] = mutual_info_classif(
        x.to_frame(), sub["DIABETES"],
        discrete_features=col not in num_set, random_state=42
    )[0]
mi = pd.Series(mi).sort_values(ascending=False)
group_of = {c: g for g, cols in predictor_groups.items() for c in cols}
colors = {"demographic": "tab:blue", "behavior": "tab:orange", "health": "tab:green"}

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(mi.index, mi.values, color=[colors[group_of[c]] for c in mi.index])
ax.legend(handles=[Patch(color=c, label=g) for g, c in colors.items()], title="Group")
ax.set(ylabel="Mutual information", title="Association with diabetes")
ax.margins(x=0.01)                       # trims empty space at the ends
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()

# Use of Generative AI

Prompt: I am performing EDA on my dataset. How can I show predictor-target relationships?\
What I did: Used the output to make an efficient for loops for the predictor-target relationships. Used it to learn what a mutual information graph and to make it.